In [1]:
import json
import pandas as pd
from datetime import datetime
from datetime import timezone
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from zoneinfo import ZoneInfo
from glob import glob
from pathlib import Path

# Trio Dataset

In [ ]:

from Scripts.insulin_trace import (
    clean_insulin_data,
    process_insulin_data
)

from Scripts.glucose_trace import (
    process_glucose_data
)

from Scripts.extract_user_profile import (
    extract_user_profile,
    save_extracted_data
) 

FOLDER_PATH = "trio_dataset/trio-oref-validation/algorithm-comparisons"

In [3]:
def cut_into_chunks(sorted_df, large_gap = pd.Timedelta(minutes=30)):
    time_diffs = sorted_df["date"].diff()

    is_large_gap = time_diffs > large_gap

    sorted_df['chunk_id'] = is_large_gap.cumsum()

    chunks = [group for _, group in sorted_df.groupby('chunk_id')]

    print(f"Chopped into {len(chunks)} chunks")

    # clean up
    for chunk in chunks:
        chunk.drop(columns=['chunk_id'], inplace=True)

    return chunks

In [ ]:
def pair_chunk_to(chunk: pd.DataFrame, candidate_chunks: list[pd.DataFrame]):
    chunk_start = chunk["date"].min()
    chunk_end = chunk["date"].max()

    if chunk_end - chunk_start < pd.Timedelta(hours=1):
        print("Not worth pairing, chunk shorter than 1 hour")
        return None    

    best_match = None
    best_overlap = pd.Timedelta(0)

    for candidate in candidate_chunks:
        candidate_start = candidate["date"].min()
        candidate_end = candidate["date"].max()

        latest_start = max(chunk_start, candidate_start)
        earliest_end = min(chunk_end, candidate_end)

        overlap = max(pd.Timedelta(0), earliest_end - latest_start)

        if overlap > best_overlap:
            best_overlap = overlap
            best_match = candidate[candidate["date"].between(latest_start, earliest_end)]

    return best_match

## Data Structure


All we are looking is the timestamped insulin amount from PumpHistoryEvent (amount property) and timestamped glucose amount from BloodGlucose (glucose property).

```Swift
struct PumpHistoryEvent: JSON, Equatable, Identifiable {
    let id: String
    let type: EventType
    let timestamp: Date
    let amount: Decimal?
    let duration: Int?
    let durationMin: Int?
    let rate: Decimal?
    let temp: TempType?
    let carbInput: Int?
    let fatInput: Int?
    let proteinInput: Int?
    let note: String?
    let isSMB: Bool?
    let isExternal: Bool?
    let isExternalInsulin: Bool?

    init(
        id: String,
        type: EventType,
        timestamp: Date,
        amount: Decimal? = nil,
        duration: Int? = nil,
        durationMin: Int? = nil,
        rate: Decimal? = nil,
        temp: TempType? = nil,
        carbInput: Int? = nil,
        fatInput: Int? = nil,
        proteinInput: Int? = nil,
        note: String? = nil,
        isSMB: Bool? = nil,
        isExternal: Bool? = nil,
        isExternalInsulin: Bool? = nil
    ) {
        self.id = id
        self.type = type
        self.timestamp = timestamp
        self.amount = amount
        self.duration = duration
        self.durationMin = durationMin
        self.rate = rate
        self.temp = temp
        self.carbInput = carbInput
        self.fatInput = fatInput
        self.proteinInput = proteinInput
        self.note = note
        self.isSMB = isSMB
        self.isExternal = isExternal
        self.isExternalInsulin = isExternalInsulin
    }
}

struct BloodGlucose: JSON, Identifiable, Hashable, Codable {
    enum Direction: String, JSON {
        case tripleUp = "TripleUp"
        case doubleUp = "DoubleUp"
        case singleUp = "SingleUp"
        case fortyFiveUp = "FortyFiveUp"
        case flat = "Flat"
        case fortyFiveDown = "FortyFiveDown"
        case singleDown = "SingleDown"
        case doubleDown = "DoubleDown"
        case tripleDown = "TripleDown"
        case none = "NONE"
        case notComputable = "NOT COMPUTABLE"
        case rateOutOfRange = "RATE OUT OF RANGE"

        ...
    }

    enum CodingKeys: String, CodingKey {
        case _id
        case idKey = "id"
        case sgv
        case direction
        case date
        case dateString
        case unfiltered
        case filtered
        case noise
        case glucose
        case type
        case activationDate
        case sessionStartDate
        case transmitterID

        ...
    }

    ...
    
    var _id: String?

    // this is a dummy property, we never set it. We have it for more flexible
    // glucose record parsing (see the `init(from decoder: Decoder)` method)
    var idKey: String?

    var sgv: Int?
    var direction: Direction?
    let date: Decimal
    let dateString: Date
    let unfiltered: Decimal?
    let filtered: Decimal?
    let noise: Int?
    var glucose: Int?
    var type: String? = nil
    var activationDate: Date? = nil
    var sessionStartDate: Date? = nil
    var transmitterID: String? = nil
    var isStateValid: Bool { sgv ?? 0 >= 39 && noise ?? 1 != 4 }

    ...
}
```

Raw json is coming from
```swift
enum JSONCompare {
    static let log = try? JsSwiftOrefComparisonLogger()
    static func logDifferences(
        function: OrefFunction,
        swift: OrefFunctionResult,
        swiftDuration: TimeInterval,
        javascript: OrefFunctionResult,
        javascriptDuration: TimeInterval,
        iobInputs: IobInputs? = nil,
        mealInputs: MealInputs? = nil,
        autosensInputs: AutosensInputs? = nil,
        determineBasalInputs: DetermineBasalInputs? = nil
    ) {
        let comparison = createComparison(
            function: function,
            swift: swift,
            swiftDuration: swiftDuration,
            javascript: javascript,
            javascriptDuration: javascriptDuration,
            iobInputs: iobInputs,
            mealInputs: mealInputs,
            autosensInputs: autosensInputs,
            determineBasalInputs: determineBasalInputs
        )

        Task {
            do {
                try await log?.logComparison(comparison: comparison)
            } catch {
                warning(.openAPS, "logComparison exception: \(error)", error: error)
            }
        }
    }
}
```

## Get insulin & glucose trace

### Single Json

In [ ]:
JSON_PATH = "trio_dataset/trio-oref-validation/algorithm-comparisons/2025-07-31/0.5.1/meal/70431AA7-98C0-4ED9-8EAB-93530905061B/0a6c019b-8d12-4541-bec0-7d9c5196862c.json"

with open(JSON_PATH) as f:
    data = json.load(f)

insulin_df = clean_insulin_data(data)
insulin_df = process_insulin_data(insulin_df)
display(insulin_df)

,date,type,unit,value,deliveryReason
0,2025-07-30 02:31:57.750999928+00:00,insulin,U,4.0,bolus
1,2025-07-30 02:37:06.102999926+00:00,insulin,U,0.0,basal
2,2025-07-30 02:46:58.059999943+00:00,insulin,U,0.0,basal
3,2025-07-30 02:56:58.605000019+00:00,insulin,U,0.0,basal
4,2025-07-30 03:07:01.321000099+00:00,insulin,U,0.0,basal
...,...,...,...,...,...
249,2025-07-31 02:12:01.006999969+00:00,insulin,U,0.25,bolus
250,2025-07-31 02:16:58.040999889+00:00,insulin,U,0.2,bolus
251,2025-07-31 02:21:58.509000063+00:00,insulin,U,0.2,bolus
252,2025-07-31 02:27:05.081000090+00:00,insulin,U,0.2,bolus


In [3]:
# insulin_df["date"] = insulin_df["date"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
# insulin_df.to_json(
#     "insulin_one.json",
#     orient="records",
#     indent=2,
# )

### For multiple json

In [ ]:

LAST_FOLDER = "meal/70431AA7-98C0-4ED9-8EAB-93530905061B" 

# LAST_FOLDER = "meal/0A9CA9B2-CD18-4AF4-8384-866278470D72"

# LAST_FOLDER = "meal/547C7B4D-D130-4A57-AFF9-C6F23D577308"

# LAST_FOLDER = "meal/681E14FF-BD54-4DFA-9C67-88A54B793F28" # not usable

# LAST_FOLDER = "meal/14CFC51D-9B72-4B05-93BC-3544FCA8D58B"

# LAST_FOLDER = "meal/673C17FA-4728-467E-9331-72EAE8FC04B4" # not usable 

# LAST_FOLDER = "meal/77F59FB6-ED9C-4989-AA86-F00FC1EAC1B1"

# LAST_FOLDER = "meal/9371E304-BC2B-49D1-91AB-C1CC82801ACF"

# LAST_FOLDER = "meal/AB00357B-DD41-4CBB-83FE-C9A43A49D34A"

# LAST_FOLDER = "meal/F349A9CB-640F-4D82-BD2E-B5089F254135"

# example folder path: downloaded_files/trio-oref-validation/algorithm-comparisons/2025-07-27/0.5.1/meal/70431AA7-98C0-4ED9-8EAB-93530905061B

Load all the json from `FOLDER_PATH/**/LAST_FOLDER/*.json` and extract the timestamped insulin and glucose like above. End result should be two dataFrames, one for insulin and one for glucose.

In [ ]:
json_files = sorted(glob(f"{FOLDER_PATH}/**/{LAST_FOLDER}/*.json", recursive=True))
print(f"Found {len(json_files)} JSON files")

Found 3493 JSON files


In [92]:
# glucose trace
glucose_list = []
insulin_list = []

for i, jf in enumerate(json_files):
    try:
        with open(jf) as f:
            data = json.load(f)

        insulin_df = clean_insulin_data(data)
        glucose_df = process_glucose_data(data)

        insulin_list.append(insulin_df)
        glucose_list.append(glucose_df)

    except MemoryError:
        print(f"MemoryError at file {i}/{len(json_files)}: {jf}")
        break
    except Exception as e:
        print(f"Error at file {i}/{len(json_files)}: {e}")
        print(f"File: {jf}")
        continue


glucose_df = pd.concat(glucose_list, ignore_index=True)
glucose_df.drop_duplicates(subset=["date"], inplace=True)

insulin_df = pd.concat(insulin_list, ignore_index=True)
insulin_df = process_insulin_data(insulin_df)
insulin_df.drop_duplicates(subset=["date", "value", "deliveryReason"], inplace=True)

display(glucose_df)
display(insulin_df)

,value,date,type,unit
0,132,2025-07-11 03:47:46.869999886+00:00,glucose,mg/dL
1,124,2025-07-11 03:42:46.842999935+00:00,glucose,mg/dL
2,116,2025-07-11 03:37:46.648999929+00:00,glucose,mg/dL
3,110,2025-07-11 03:32:47.173000097+00:00,glucose,mg/dL
4,107,2025-07-11 03:27:46.676000118+00:00,glucose,mg/dL
...,...,...,...,...
214821,110,2025-11-20 16:03:00.803999901+00:00,glucose,mg/dL
214822,108,2025-11-20 15:58:00.469000101+00:00,glucose,mg/dL
214823,107,2025-11-20 15:53:00.964999914+00:00,glucose,mg/dL
214885,113,2025-11-20 16:48:00.220000029+00:00,glucose,mg/dL


,date,type,unit,value,deliveryReason
0,2025-07-10 03:13:08.500000+00:00,insulin,U,0.25,bolus
1,2025-07-10 03:18:02.283999920+00:00,insulin,U,0.0,basal
2,2025-07-10 03:18:06.418999910+00:00,insulin,U,0.25,bolus
3,2025-07-10 03:23:04.328000069+00:00,insulin,U,0.0,basal
4,2025-07-10 03:23:05.448999882+00:00,insulin,U,0.45,bolus
...,...,...,...,...,...
9812,2025-11-20 16:33:18.825000048+00:00,insulin,U,0.05,bolus
9813,2025-11-20 16:38:15.049999952+00:00,insulin,U,0.0126,basal
9814,2025-11-20 16:43:16.631999969+00:00,insulin,U,0.0687,basal
9815,2025-11-20 16:48:25.726000071+00:00,insulin,U,0.5,basal


### Cut and pair insulin chunks with glucose chunks

In [93]:
glucose_df.sort_values("date", inplace=True, ignore_index=True)
glucose_chunks = cut_into_chunks(glucose_df)

Chopped into 4 chunks


In [94]:
insulin_df.sort_values("date", inplace=True, ignore_index=True)
insulin_chunks = cut_into_chunks(insulin_df)

Chopped into 67 chunks


In [95]:
# find matching for insulin
combined_results: list[pd.DataFrame] = []
for chunk in insulin_chunks:
    matching_glucose: pd.DataFrame = pair_chunk_to(chunk, glucose_chunks)
    if matching_glucose is not None:
        combined = pd.concat([chunk, matching_glucose], ignore_index=True)
        combined.sort_values("date", inplace=True, ignore_index=True)
        combined_results.append(combined)

print(f"Paired {len(combined_results)} chunks")

Not worth pairing, chunk shorter than 1 hour
Not worth pairing, chunk shorter than 1 hour
Not worth pairing, chunk shorter than 1 hour
Not worth pairing, chunk shorter than 1 hour
Not worth pairing, chunk shorter than 1 hour
Not worth pairing, chunk shorter than 1 hour
Paired 63 chunks


In [96]:
user_id = Path(LAST_FOLDER).name
save_dir = Path("traces") / user_id
save_dir.mkdir(parents=True, exist_ok=True)

for chunk in combined_results:
    start_date = chunk['date'].iloc[0].strftime('%Y-%m-%dT%H:%M:%SZ')
    end_date = chunk['date'].iloc[-1].strftime('%Y-%m-%dT%H:%M:%SZ')
    chunk['date'] = chunk['date'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    chunk.to_json(
        save_dir / f"t1d_data_{start_date}_{end_date}.json",
        orient="records",
        indent=2,
    )

### Visualize

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Use normalized combined schema: date/type/value/deliveryReason
plot_df = combined_df[combined_df["type"] == "insulin"].copy()
plot_df["dt"] = pd.to_datetime(plot_df["date"], utc=True)

bolus_df = plot_df[plot_df["deliveryReason"] == "bolus"]
basal_df = plot_df[plot_df["deliveryReason"] == "basal"]

fig, ax = plt.subplots(figsize=(16, 5))

# Basal insulin as smaller blue points
ax.scatter(basal_df["dt"], basal_df["value"], s=6, color="tab:blue", alpha=0.5, label="Basal")

# Bolus insulin as larger red points
ax.scatter(bolus_df["dt"], bolus_df["value"], s=20, color="tab:red", alpha=0.8, label="Bolus", zorder=5)

ax.set_xlabel("Timestamp (UTC)")
ax.set_ylabel("Insulin (U)")
ax.set_title("Absolute Insulin Delivery Over Time")
ax.legend(loc="upper right")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M", tz=ZoneInfo("UTC")))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [50]:
# import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
# from zoneinfo import ZoneInfo

# # Use normalized combined schema and convert to Pacific for windowed plotting
# plot_df = combined_df[combined_df["type"] == "insulin"].copy()
# plot_df["dt"] = pd.to_datetime(plot_df["date"])

# # Filter to desired date range (full day: midnight to midnight Pacific timezone)
# start_date = pd.Timestamp("2025-07-19 00:00:00", tz="US/Pacific")
# end_date = pd.Timestamp("2025-07-20 00:00:00", tz="US/Pacific")
# plot_df = plot_df[(plot_df["dt"] >= start_date) & (plot_df["dt"] < end_date)]

# bolus_df = plot_df[plot_df["deliveryReason"] == "bolus"]
# basal_df = plot_df[plot_df["deliveryReason"] == "basal"]

# fig, ax = plt.subplots(figsize=(16, 5))

# # Basal insulin as smaller blue points
# ax.scatter(basal_df["dt"], basal_df["value"], s=8, color="tab:blue", alpha=0.5, label="Basal")

# # Bolus insulin as larger red points
# ax.scatter(bolus_df["dt"], bolus_df["value"], s=20, color="tab:red", alpha=0.8, label="Bolus", zorder=5)

# ax.set_xlabel("Timestamp (Pacific Time)")
# ax.set_ylabel("Insulin (U)")
# ax.set_title("Absolute Insulin Delivery Over Time (Pacific Timezone)")
# ax.legend(loc="upper right")
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M", tz=ZoneInfo("US/Pacific")))
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

In [51]:
# print("A day's insulin: ", plot_df["value"].sum(), "U")

In [52]:
# start_date = pd.Timestamp("2025-07-15 00:00:00", tz="US/Pacific").tz_convert("UTC")
# end_date = pd.Timestamp("2025-08-10 00:00:00", tz="US/Pacific").tz_convert("UTC")

# start_str = start_date.strftime('%Y-%m-%dT%H:%M:%SZ')
# end_str = end_date.strftime('%Y-%m-%dT%H:%M:%SZ')

# one_day_df = combined_df[(combined_df["date"] >= start_str) & (combined_df["date"] < end_str)]
# one_day_df.to_json(
#     f"t1d_data_2025-07-19.json",
#     orient="records",
#     indent=2,
# )
# display(one_day_df)

## Get the user profile

In [ ]:
LAST_FOLDER = "determineBasal/70431AA7-98C0-4ED9-8EAB-93530905061B" #

# LAST_FOLDER = "determineBasal/0A9CA9B2-CD18-4AF4-8384-866278470D72"

# LAST_FOLDER = "determineBasal/547C7B4D-D130-4A57-AFF9-C6F23D577308"

# LAST_FOLDER = "determineBasal/681E14FF-BD54-4DFA-9C67-88A54B793F28" # no profile data usable

# LAST_FOLDER = "determineBasal/14CFC51D-9B72-4B05-93BC-3544FCA8D58B"

# LAST_FOLDER = "determineBasal/673C17FA-4728-467E-9331-72EAE8FC04B4" # not usable 

# LAST_FOLDER = "determineBasal/77F59FB6-ED9C-4989-AA86-F00FC1EAC1B1"

# LAST_FOLDER = "determineBasal/9371E304-BC2B-49D1-91AB-C1CC82801ACF"

# LAST_FOLDER = "determineBasal/AB00357B-DD41-4CBB-83FE-C9A43A49D34A"

# LAST_FOLDER = "determineBasal/F349A9CB-640F-4D82-BD2E-B5089F254135"


json_files = sorted(glob(f"{FOLDER_PATH}/**/{LAST_FOLDER}/*.json", recursive=True))
print(f"Found {len(json_files)} JSON files")

Found 3518 JSON files


In [35]:
extracted_data = None
for i, jf in enumerate(json_files):
    try:
        with open(jf) as f:
            data = json.load(f)
        
        ok, extracted_data = extract_user_profile(data, extracted_data)
        if ok:
            print("Found all necessary user profile data! Stopping search.")
            break

    except Exception as e:
        print(f"Error at file {i}/{len(json_files)}: {e}")
        print(f"File: {jf}")
        continue

Found all necessary user profile data! Stopping search.


In [ ]:
baseFolder = Path(LAST_FOLDER).name 
virtualUserDir = Path("VirtualUsers")

In [37]:
if extracted_data is not None:  
    save_extracted_data(extracted_data, virtualUserDir / baseFolder)
else:
    print("Failed to find any user profile data.")

Saved preferences.json
Saved carbs_ratios.json
Saved basal_profile.json
Saved bg_targets.json
Saved insulin_sensitivities.json
Saved settings.json
Saved temptargets.json
